In [8]:
# !pip install langchain langchain-community faiss-cpu sentence-transformers langchain-huggingface datasets

   ---------------------------------------- 0.0/529.0 kB ? eta -:--:--
   --------------------------------------- 529.0/529.0 kB 16.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/27.3 MB ? eta -:--:--
   - -------------------------------------- 1.0/27.3 MB 4.5 MB/s eta 0:00:06
   -- ------------------------------------- 1.6/27.3 MB 3.3 MB/s eta 0:00:08
   -- ------------------------------------- 1.8/27.3 MB 3.1 MB/s eta 0:00:09
   --- ------------------------------------ 2.4/27.3 MB 2.9 MB/s eta 0:00:09
   ---- ----------------------------------- 2.9/27.3 MB 2.6 MB/s eta 0:00:10
   ---- ----------------------------------- 3.4/27.3 MB 2.6 MB/s eta 0:00:10
   ----- ---------------------------------- 3.9/27.3 MB 2.6 MB/s eta 0:00:10
   ------ --------------------------------- 4.5/27.3 MB 2.6 MB/s eta 0:00:09
   ------- -------------------------------- 5.0/27.3 MB 2.5 MB/s eta 0:00:09
   -------- ------------------------------- 5.5/27.3 MB 2.5 MB/s eta 0:00:09
   ------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires packaging<25,>=20, but you have packaging 26.0 which is incompatible.
streamlit 1.45.1 requires pandas<3,>=1.4.0, but you have pandas 3.0.2 which is incompatible.
streamlit 1.45.1 requires pillow<12,>=7.1.0, but you have pillow 12.1.1 which is incompatible.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7.34.1 which is incompatible.


In [2]:
# !pip uninstall -y pandas pyarrow datasets
# !conda install -y -c conda-forge --force-reinstall pandas pyarrow datasets

Found existing installation: pandas 2.2.3
Uninstalling pandas-2.2.3:
  Successfully uninstalled pandas-2.2.3
Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
Found existing installation: datasets 4.8.5
Uninstalling datasets-4.8.5:
  Successfully uninstalled datasets-4.8.5
Jupyter detected...
3 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: C:\Users\justi\anaconda3

  added / updated specs:
    - datasets
    - pandas
    - pyarrow


The following NEW packages will be INSTALLED:

  datasets           conda-forge/noarch::datasets-2.14.4-pyhd8ed1ab_0 
  pyarrow            pkgs/main/win-64::pyarrow-19.0.0-py313h5da7b33_1 



Preparing transaction: done
Verifying transaction: done
Executing transaction: done




==> WARNING: A newer version of conda exists. <==
    current version: 25.7.0
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda




In [3]:
# !pip install --upgrade transformers trl peft

  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached pyarrow-24.0.0-cp313-cp313-win_amd64.whl.metadata (3.0 kB)
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/10.6 MB 9.8 MB/s eta 0:00:01
   ------- -------------------------------- 2.1/10.6 MB 5.3 MB/s eta 0:00:02
   --------- ------------------------------ 2.6/10.6 MB 4.2 MB/s eta 0:00:02
   ----------- ---------------------------- 3.1/10.6 MB 3.7 MB/s eta 0:00:03
   ------------- -------------------------- 3.7/10.6 MB 3.4 MB/s eta 0:00:03
   ------------- -------------------------- 3.7/10.6 MB 3.4 MB/s eta 0:00:03
   ------------- -------------------------- 3.7/10.6 MB 3.4 MB/s eta 0:00:03
   --------------- ------------------------ 4.2/10.6 MB 2.4 MB/s eta 0:00:03
   ---------------- ----------------------- 4.5/10.6 MB 2.4 MB/s eta 0:00:03
   ------------------ --------------------- 5.0/10.6 MB 2.4 MB/s eta 0:00:03
   -------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires packaging<25,>=20, but you have packaging 26.0 which is incompatible.
streamlit 1.45.1 requires pillow<12,>=7.1.0, but you have pillow 12.1.1 which is incompatible.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7.34.1 which is incompatible.


# CUSTOM PIPELINE CONSTRUCTION

## TOKENIZER

In [1]:
import pandas as pd

class AtakapaTokenizer:
    def __init__(self, lexicon='atakapa_lexicon_final.xlsx', morph_rules='atakapa_morphology_UTF8.xlsx'):
        self.lexicon = pd.read_excel(lexicon)
        self.morph_rules = pd.read_excel(morph_rules)

        clean_apq = self.lexicon['apq'].astype(str).str.strip().tolist()
        norm_roots = set([self._normalize_text(root) for root in clean_apq])
        strip_roots = set([self._remove_accents(root) for root in norm_roots])

        
        self.m_word_exp = sorted([root for root in strip_roots if " " in root], key=len, reverse=True)
        self.valid_roots = {root.replace(' ', '_') for root in strip_roots}
        
        # raw_prefixes_list = self.morph_rules[self.morph_rules['affix_type'] == 'Prefix']['morpheme_action'].tolist()
        # raw_prefixes =[self._normalize_text(p) for p in raw_prefixes_list]
        # self.prefixes = sorted(raw_prefixes, key=len, reverse=True)

        prefix_df = self.morph_rules[self.morph_rules['affix_type'] == 'Prefix']
        self.prefixes = {}
        for _, row in prefix_df.iterrows():
            norm_prefix = self._normalize_text(row['morpheme_action'])
            strip_prefix = self._remove_accents(row['morpheme_action'])
            self.prefixes[strip_prefix] = int(row['slot_position'])
        self.prefix_list = sorted(list(self.prefixes.keys()), key=len, reverse=True)
        
        # raw_suffixes_list = self.morph_rules[self.morph_rules['affix_type'] == 'Suffix']['morpheme_action'].tolist()
        # raw_suffixes =[self._normalize_text(p) for p in raw_suffixes_list]
        # self.suffixes = sorted(raw_suffixes, key=len, reverse=True)

        suffix_df = self.morph_rules[self.morph_rules['affix_type'] == 'Suffix']
        self.suffixes = {}
        for _, row in suffix_df.iterrows():
            norm_suffix = self._normalize_text(row['morpheme_action'])
            strip_suffix = self._remove_accents(row['morpheme_action'])
            self.suffixes[strip_suffix] = int(row['slot_position'])
            
        self.suffix_list = sorted(list(self.suffixes.keys()), key=len, reverse=True)
        
        self.vocab = self._build_vocab()
        self.vocab_inv = {value : key for key, value in self.vocab.items()}

    def _remove_accents(self, text):
        '''Creates a purely unaccented string'''
        text = str(text).lower().strip()
        for p in ['·', 'ˑ', 'ː', '́', '̀']:
            text = text.replace(p, '')
       
        replacements = {
            'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
            'à': 'a', 'è': 'e', 'ì': 'i', 'ò': 'o', 'ù': 'u',
        }
        
        for accented, flat in replacements.items():
            text = text.replace(accented, flat)
            
        return text

    
    def _normalize_text(self, text):
        '''Removes punctuation and applies phonemic representation to phonetic realizations'''
        text = str(text).lower().strip()
        for p in ['.', ',', '?', '!', '…', '"', "'",';']:
            text = text.replace(p, '')
        replacements = {
            'ɬ': 'ƛ', 'ł': 'ƛ'
        }
        for phonetic, phonemic in replacements.items():
            text = text.replace(phonetic, phonemic)
            
        return text

        
    def _build_vocab(self):
        '''Builds vocabulary from roots, prefixes and suffixes'''
        base_dict = {'[PAD]':0 , '[UNK]':1, '[BOS]':2, '[EOS]':3}
        i = 4
        sets = [self.valid_roots, self.prefixes, self.suffixes]
        
        for s in sets:
            for entry in s:
                if entry not in base_dict:
                    base_dict[entry] = i
                    i += 1
        return base_dict


    def _strict_parse(self, current_str, memo=None):
        '''Recursively explores ALL affix paths and returns every valid combination.'''
        if memo is None:
            memo = {}
        if current_str in memo:
            return memo[current_str]
            
        valid_paths = []

        # 1. Base Case
        if current_str in self.valid_roots:
            valid_paths.append([current_str])
                    
        # 2. Suffixes
        for suffix in self.suffixes:
            stripped_suffix = suffix.replace('-', '')
            if current_str.endswith(stripped_suffix) and len(stripped_suffix) > 0:
                sub_paths = self._strict_parse(current_str[:-len(stripped_suffix)], memo)
                for p in sub_paths:
                    valid_paths.append(p + [suffix])
                    
        # 3. Prefixes
        for prefix in self.prefixes:
            stripped_prefix = prefix.replace('-', '')
            if current_str.startswith(stripped_prefix) and len(stripped_prefix) > 0:
                sub_paths = self._strict_parse(current_str[len(stripped_prefix):], memo)
                for p in sub_paths:
                    valid_paths.append([prefix] + p)
                    
        # 4. Unlimited Compounding
        for pivot in range(1, len(current_str)):
            left_root = current_str[:pivot]
            if left_root in self.valid_roots:
                sub_paths = self._strict_parse(current_str[pivot:], memo)
                for p in sub_paths:
                    valid_paths.append([left_root] + p)
                    
        # Save to memo to prevent lag
        memo[current_str] = valid_paths
        return valid_paths
    # def _strict_parse(self, current_str):
    #     """Recursively explores affix paths to find a perfect dictionary match."""
    #     # Base Case: The string is a perfect dictionary root
    #     if current_str in self.valid_roots:
    #         return [current_str]
            
    #     # Branch 1: Try Suffixes
    #     for suffix in self.suffixes:
    #         stripped_suffix = suffix.replace('-', '')
    #         if current_str.endswith(stripped_suffix):
    #             # Recursively check the remainder of the word
    #             path = self._strict_parse(current_str[:-len(stripped_suffix)])
    #             if path:
    #                 return path + [suffix]
                    
        # # Branch 2: Try Prefixes
        # for prefix in self.prefixes:
        #     stripped_prefix = prefix.replace('-', '')
        #     if current_str.startswith(stripped_prefix):
        #         # Recursively check the remainder of the word
        #         path = self._strict_parse(current_str[len(stripped_prefix):])
        #         if path:
        #             return [prefix] + path
                    
        # # Branch 3: Unlimited Compounding
        # for pivot in range(1, len(current_str)):
        #     left_root = current_str[:pivot]
        #     # If the left side is a valid root, recursively parse the right side
        #     if left_root in self.valid_roots:
        #         right_path = self._strict_parse(current_str[pivot:])
        #         if right_path:
        #             return [left_root] + right_path
                    
        # # Dead End: This path did not lead to valid roots
        # return None

    def _greedy_fallback(self, word):
        '''Peels known affixes off an unknown root as a best-effort fallback.'''
        tokens = []
        current_str = word
        
        # Peel all prefixes
        prefix_found = True
        while prefix_found:
            prefix_found = False
            for prefix in self.prefixes:
                stripped_prefix = prefix.replace('-', '')
                if current_str.startswith(stripped_prefix):
                    prefix_found = True
                    tokens.append(prefix)
                    current_str = current_str[len(stripped_prefix):]
                    break
                    
        # Peel all suffixes
        temp_suffixes = []
        suffix_found = True
        while suffix_found:
            suffix_found = False
            for suffix in self.suffixes:
                stripped_suffix = suffix.replace('-', '')
                if current_str.endswith(stripped_suffix):
                    suffix_found = True
                    temp_suffixes.insert(0, suffix)
                    current_str = current_str[:-len(stripped_suffix)]
                    break
                    
        # Assemble (Prefixes + Unknown Core + Suffixes)
        if current_str:
            tokens.append(current_str)
        tokens.extend(temp_suffixes)
        
        return tokens

    def _align(self, original_word, clean_segments):
        '''Applies the unaccented morphological boundaries back onto accented string'''
        aligned = []
        orig_idx = 0
        removed_chars = set(['·', 'ˑ', 'ː', '́', '̀'])

        for segment in clean_segments:
            clean_str = segment.replace('-', '')
            aligned_segment = ""
            chars_matched = 0
            
            # Rebuild the string using lengths, ignoring 'invisible' marks
            while chars_matched < len(clean_str) and orig_idx < len(original_word):
                c = original_word[orig_idx]
                aligned_segment += c
                orig_idx += 1
                
                if c in removed_chars:
                    pass # Free character, doesn't count against segment length
                else:
                    chars_matched += 1 # Normal or precomposed letter ('á') counts as 1
                    
            # Catch trailing accents/lengths attached to the end of this morpheme
            while orig_idx < len(original_word) and original_word[orig_idx] in removed_chars:
                aligned_segment += original_word[orig_idx]
                orig_idx += 1
                
            # Restore dictionary hyphens 
            if segment.startswith('-') and not aligned_segment.startswith('-'):
                aligned_segment = '-' + aligned_segment
            if segment.endswith('-') and not aligned_segment.endswith('-'):
                aligned_segment = aligned_segment + '-'
                
            aligned.append(aligned_segment)
            
        return aligned


    def _segment_word(self, word):
        """Routes words through parsing logic using the DUAL-LAYER method."""
        # 1. Strip accents for perfect dictionary matching
        clean_word = self._remove_accents(word)
        
        # 2. Parse the clean word
        all_paths = self._strict_parse(clean_word)
        
        best_clean_path = None
        if all_paths:
            grammatical_paths = [p for p in all_paths if self._validate_morphotactics(p)]
            if grammatical_paths:                
                grammatical_paths.sort(key=len)
                best_clean_path = grammatical_paths[0]
                
        if not best_clean_path:
            best_clean_path = self._greedy_fallback(clean_word)
            
        # 3. Align the perfectly parsed clean segments back onto the accented surface word
        return self._align(word, best_clean_path)
        
    # def _segment_word(self, word):
    #     all_paths = self._strict_parse(word)
        
    #     if all_paths:
    #         grammatical_paths = [p for p in all_paths if self._validate_morphotactics(p)]
            
    #         if grammatical_paths:                
    #             grammatical_paths.sort(key=len)
    #             return grammatical_paths[0]
            
    #     return self._greedy_fallback(word)
        
    def _validate_morphotactics(self, path):
        """
        Validates that morphemes follow strict templatic slot order.
        """
        positions = []
        for token in path:
            if token in self.prefixes:
                positions.append(self.prefixes[token])
            elif token in self.suffixes:
                positions.append(self.suffixes[token])
            else:
                positions.append(0) # Roots are 0

        for i in range(len(positions) - 1):
            # 1. Enforce Templatic Slots (Left-to-Right Outward flow)
            if positions[i] > positions[i+1]:
                return False
                
            # 2. Prevent Slot Collisions
            if positions[i] == positions[i+1] and positions[i] != 0:
                return False

            # 3. EXPLICIT TERMINAL LOCK (Optional but highly recommended)
            # If Slot 5 (or whatever your max slot is) is your phrase terminal marker,
            # explicitly forbid anything from ever coming after it.
            # TERMINAL_SLOT = 5 
            # if positions[i] == TERMINAL_SLOT:
            #     return False # Fails because a terminal slot cannot have a [i+1] neighbor
                
        return True
    # def _segment_word(self, word):
    #         """Routes words through the parsing logic, prioritizing the cleanest chunks."""
    #         # Gather ALL mathematically valid paths
    #         all_paths = self._strict_parse(word)
            
    #         if all_paths:
    #             # Sort paths by length (fewest morphemes wins!)
    #             all_paths.sort(key=len)
    #             return all_paths[0]
                
    #         # If no perfect path exists, fall back to peeling
    #         return self._greedy_fallback(word)
    # def _segment_word(self, word):
    #     """The public method that routes words through the parsing logic."""
    #     # Try to find a perfect path using the dictionary
    #     strict_path = self._strict_parse(word)
    #     if strict_path:
    #         return strict_path
            
    #     # If the word contains unknown roots, fall back to peeling
    #     return self._greedy_fallback(word)
    def encode(self, text):
        token_ids = []
        token_ids.append(self.vocab['[BOS]'])

        text = text.lower()
        text = self._normalize_text(text)
        
        for mwe in self.m_word_exp:
            if mwe in text:
                text = text.replace(mwe, mwe.replace(' ','_'))
        
        words = text.split()
        for word in words:
            morphemes = self._segment_word(word)
            for morpheme in morphemes:
                morpheme_id = self.vocab.get(morpheme, self.vocab['[UNK]'])
                token_ids.append(morpheme_id)
        
        token_ids.append(self.vocab['[EOS]'])
        return token_ids

    
    def decode(self, token_ids):
        morpheme_strings = []
        ctrl_ids = [0,2,3]
        
        for ids in token_ids:
            if ids in ctrl_ids:
                continue
            elif ids in self.vocab_inv:
                morpheme_strings.append(self.vocab_inv[ids])
        
        final_string = ""
        for m in morpheme_strings:
            m = m.replace('_',' ')

            if final_string == "":
                final_string += m
            elif m.startswith('-'):
                final_string += m.replace('-','')
            elif final_string.endswith('-'):
                final_string = final_string[:-1] + m
            else:
                final_string += " " + m
        
        return final_string

In [2]:
# Initialize the Tokenizer (Make sure the files are in the same folder as this notebook)
tokenizer = AtakapaTokenizer(
    lexicon='atakapa_lexicon_final.xlsx', 
    morph_rules='atakapa_morphology_UTF8.xlsx'
)

# Test Data
test_sentence = "wáŋhokpémkin" 

# Process
encoded_ids = tokenizer.encode(test_sentence)
decoded_text = tokenizer.decode(encoded_ids)

# Print Diagnostics
print(f"ORIGINAL TEXT:  {test_sentence}")
print(f"ENCODED IDS:    {encoded_ids}")

print("ID MAPPING:     ", end="")
for token_id in encoded_ids:
    string_val = tokenizer.vocab_inv.get(token_id, "UNKNOWN")
    print(f"[{string_val} -> {token_id}] ", end="")

print(f"\n\nDECODED TEXT:   {decoded_text}")

ORIGINAL TEXT:  wáŋhokpémkin
ENCODED IDS:    [2, 1, 1782, 3]
ID MAPPING:     [[BOS] -> 2] [[UNK] -> 1] [-kin -> 1782] [[EOS] -> 3] 

DECODED TEXT:   [UNK]kin


In [3]:
print("šokšakyikš" in tokenizer.valid_roots)
print("hok-" in tokenizer.prefixes)

True
True


In [4]:
import pandas as pd
from collections import Counter

def parse_gloss_chunk(chunk, tokenizer):
    """Splits human glosses into individual morphemes with attached hyphens"""
    chunk = chunk.replace('=', '-')
    
    if '-' not in chunk:
        return [chunk]
        
    raw_parts = chunk.split('-')
    parts = [p for p in raw_parts if p.lower() not in {'ø', ''}]
    
    if not parts:
        return []

    root_idx = -1
    for i, part in enumerate(parts):
        if tokenizer._normalize_text(part) in tokenizer.valid_roots:
            root_idx = i
            break
            
    if root_idx == -1:
        longest_len = -1
        for i, part in enumerate(parts):
            if len(part) > longest_len:
                longest_len = len(part)
                root_idx = i
                
    morphemes = []
    for i, part in enumerate(parts):
        if i < root_idx:
            morphemes.append(part + '-')
        elif i > root_idx:
            morphemes.append('-' + part)
        else:
            morphemes.append(part)
            
    return morphemes


def evaluate_tokenizer(tokenizer, file_path='atakapa_corpus_with_fixed_column.xlsx'):
    print(f"Loading and cleaning gold standard data from {file_path}...\n")
    
    try:
        if file_path.endswith('.csv'):
            df = pd.read_csv(file_path, encoding='utf-8')
        else:
            df = pd.read_excel(file_path)
    except FileNotFoundError:
        print(f"Error: Could not find '{file_path}'.")
        return

    df = df.dropna(subset=['Modern_Ortho', 'Gloss.5_Fixed'])
    null_morphemes = {'ø', 'ø-', '-ø', '-ø-', 'Ø', 'Ø-', '-Ø'}
    gold_standard_data = []
    
    for index, row in df.iterrows():
        example_num = row['Example_Number']
        sentence = str(row['Modern_Ortho']).strip()
        raw_gloss = str(row['Gloss.5_Fixed']).strip()
        
        raw_morpheme_list = raw_gloss.split()
        clean_chunks = [
            m.strip() for m in raw_morpheme_list 
            if m.strip() and m.strip() not in null_morphemes
        ]
                
        if sentence and clean_chunks:
            gold_standard_data.append((example_num, sentence, clean_chunks))

    total_sentences = len(gold_standard_data)

    exact_matches = 0
    total_tp = 0
    total_fp = 0
    total_fn = 0
    error_log = []

    for example_num, sentence, expected_chunks in gold_standard_data:
        clean_sentence = tokenizer._normalize_text(sentence)
        
        # 1. Parse Expected Morphemes
        clean_expected = []
        for chunk in expected_chunks:
            clean_chunk = tokenizer._normalize_text(chunk)
            parsed_morphemes = parse_gloss_chunk(clean_chunk, tokenizer)
            clean_expected.extend(parsed_morphemes)
            
        # 2. Parse Predicted Morphemes 
        predicted_morphemes = []
        for word in clean_sentence.split():
            predicted_morphemes.extend(tokenizer._segment_word(word))
            
        # ---------------------------------------------------------
        # NEW: Strip hyphens for pure segment comparison
        # ---------------------------------------------------------
        clean_expected = [m.replace('-', '') for m in clean_expected]
        predicted_morphemes = [m.replace('-', '') for m in predicted_morphemes]
        
        # Exact Match Logic
        if predicted_morphemes == clean_expected:
            exact_matches += 1
        else:
            error_log.append({
                'number': example_num,
                'word': clean_sentence,
                'expected': clean_expected,
                'predicted': predicted_morphemes
            })
            
        # Morpheme-Level Metrics Logic
        expected_counts = Counter(clean_expected)
        predicted_counts = Counter(predicted_morphemes)
        
        tp = sum((expected_counts & predicted_counts).values())
        fp = sum((predicted_counts - expected_counts).values())
        fn = sum((expected_counts - predicted_counts).values())
        
        total_tp += tp
        total_fp += fp
        total_fn += fn

    exact_match_acc = (exact_matches / total_sentences) * 100 if total_sentences else 0
    precision = (total_tp / (total_tp + total_fp)) if (total_tp + total_fp) else 0
    recall = (total_tp / (total_tp + total_fn)) if (total_tp + total_fn) else 0
    f1_score = (2 * (precision * recall) / (precision + recall)) if (precision + recall) else 0

    print("==========================================")
    print("TOKENIZER EVALUATION REPORT")
    print("==========================================")
    print(f"Total Sentences Tested: {total_sentences}")
    print(f"Exact Match Accuracy: {exact_match_acc:.2f}%\n")
    
    print("MORPHEME-LEVEL METRICS:")
    print(f"Precision: {precision:.4f}  (Are the predicted morphemes correct?)")
    print(f"Recall:    {recall:.4f}  (Did it find all the expected morphemes?)")
    print(f"F1-Score:  {f1_score:.4f}  (Overall performance)\n")
    
    if error_log:
        pd.DataFrame(error_log).to_excel('error_log.xlsx')
        print("==========================================")
        print(f"ERROR LOG ({len(error_log)} failures):")
        print("==========================================")
        for error in error_log:
            print(f"Number:  {error['number']}")
            print(f"Sentence:  {error['word']}")
            print(f"Expected:  {error['expected']}")
            print(f"Predicted: {error['predicted']}")
            print("-" * 40)
    else:
        print("No errors detected! Perfect segmentation.")

# Run the execution
evaluate_tokenizer(tokenizer, file_path='atakapa_corpus_with_fixed_column.xlsx')

Loading and cleaning gold standard data from atakapa_corpus_with_fixed_column.xlsx...

TOKENIZER EVALUATION REPORT
Total Sentences Tested: 420
Exact Match Accuracy: 60.95%

MORPHEME-LEVEL METRICS:
Precision: 0.8366  (Are the predicted morphemes correct?)
Recall:    0.8318  (Did it find all the expected morphemes?)
F1-Score:  0.8342  (Overall performance)

ERROR LOG (164 failures):
Number:  5
Sentence:  há·š šokòy
Expected:  ['há', 'iš', 'šokòy']
Predicted: ['há·', 'š', 'šokòy']
----------------------------------------
Number:  6
Sentence:  hikí·ke išà·k
Expected:  ['hikí', 'ike', 'išà·k']
Predicted: ['hik', 'í·ke', 'išà·k']
----------------------------------------
Number:  7
Sentence:  tákapa kìš yukhíti núŋkin ké·tn̥ tat
Expected:  ['tákapa', 'kìš', 'yukhíti', 'núŋ', 'kin', 'ké·t', 'n̥', 'ta', 't']
Predicted: ['tákapa', 'kìš', 'yukhíti', 'núŋ', 'kin', 'ké·t', 'n̥', 'tat']
----------------------------------------
Number:  21
Sentence:  šokšakyíkšya šokwákn a·l payókya 
Expected:  ['šok

In [ ]:
'''DEPRECIATED VERSION'''

# import pandas as pd

# class AtakapaTokenizer:
#     def __init__(self, lexicon='atakapa_lexicon_final.xlsx', morph_rules='atakapa_morphology_UTF8.xlsx'):
#         self.lexicon = pd.read_excel(lexicon)
#         self.morph_rules = pd.read_excel(morph_rules)

#         clean_apq = self.lexicon['apq'].astype(str).str.strip().tolist()
#         temp_roots = set([self._normalize_text(root) for root in clean_apq])
        
#         self.m_word_exp = sorted([root for root in temp_roots if " " in root], key=len, reverse=True)
#         self.valid_roots = {root.replace(' ', '_') for root in temp_roots}
        
#         raw_prefixes_list = self.morph_rules[self.morph_rules['affix_type'] == 'Prefix']['morpheme_action'].tolist()
#         raw_prefixes =[self._normalize_text(p) for p in raw_prefixes_list]
#         self.prefixes = sorted(raw_prefixes, key=len, reverse=True)

#         raw_suffixes_list = self.morph_rules[self.morph_rules['affix_type'] == 'Suffix']['morpheme_action'].tolist()
#         raw_suffixes =[self._normalize_text(p) for p in raw_suffixes_list]
#         self.suffixes = sorted(raw_suffixes, key=len, reverse=True)

#         self.vocab = self._build_vocab()
#         self.vocab_inv = {value : key for key, value in self.vocab.items()}

#     def _normalize_text(self, text):
#         text = str(text).lower().strip()
        
#         text = text.replace('·', '').replace('ˑ', '').replace('ː', '').replace('́', '').replace('̀', '')
#         replacements = {
#             'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
#             'à': 'a', 'è': 'e', 'ì': 'i', 'ò': 'o', 'ù': 'u'
#         }
#         for accented, flat in replacements.items():
#             text = text.replace(accented, flat)
            
#         return text

        
#     def _build_vocab(self):
#         base_dict = {'[PAD]':0 , '[UNK]':1, '[BOS]':2, '[EOS]':3}
#         i = 4
#         sets = [self.valid_roots, self.prefixes, self.suffixes]
        
#         for s in sets:
#             for entry in s:
#                 if entry not in base_dict:
#                     base_dict[entry] = i
#                     i += 1
#         return base_dict

    
#     def _segment_word(self, word):
#         if word in self.valid_roots:
#             return [word]
#         tokens = []
#         current_str = word
#         '''
#         STRIPPING SUFFIXES
#         '''
#         temp_suffixes = []
#         suffix_found = True
#         while suffix_found:
#             if current_str in self.valid_roots:
#                 break
                
#             suffix_found = False    
#             '''
#             LOOKAHEAD FOR ROOT
#             '''         
#             for suffix in self.suffixes:
#                 stripped_suffix = suffix.replace('-','')
#                 if current_str.endswith(stripped_suffix):
#                     test_str = current_str[:-len(stripped_suffix)]

#                     if test_str in self.valid_roots:
#                         suffix_found = True
#                         temp_suffixes.insert(0, suffix)
#                         current_str = test_str
#                         break
#             '''
#             GREEDY FALLBACK
#             '''
#             if not suffix_found:
#                 for suffix in self.suffixes:
#                     stripped_suffix = suffix.replace('-','')
#                     if current_str.endswith(stripped_suffix):
#                         suffix_found = True
#                         temp_suffixes.insert(0, suffix)
#                         current_str = current_str[:-len(stripped_suffix)] 
#         '''
#         STRIPPING PREFIXES 
#         '''
#         prefix_found = True
#         while prefix_found:
#             if current_str in self.valid_roots:
#                 break
                
#             prefix_found = False
#             '''
#             LOOKAHEAD FOR ROOTS
#             '''
#             for prefix in self.prefixes:
#                 stripped_prefix = prefix.replace('-','')
#                 if current_str.startswith(stripped_prefix):
#                     test_str = current_str[len(stripped_prefix):]
#                     if test_str in self.valid_roots:
#                         prefix_found = True
#                         tokens.append(prefix)
#                         current_str = test_str
#                         break
#             '''
#             GREEDY FALLBACK
#             '''       
#             if not prefix_found:
#                 for prefix in self.prefixes:
#                     stripped_prefix = prefix.replace('-','')
#                     if current_str.startswith(stripped_prefix):
#                         prefix_found = True
#                         tokens.append(prefix)
#                         current_str = current_str[len(stripped_prefix):]
#                         break
#         '''
#         VALIDATION AND COMPOUNDING
#         '''                   
#         if current_str in self.valid_roots:
#             tokens.extend([current_str] + temp_suffixes)
#         else:
#             compound_found = False

#             for pivot in range(1,len(current_str)):
#                 left_root = current_str[:pivot]
#                 right_root = current_str[pivot:]

#                 if left_root in self.valid_roots and right_root in self.valid_roots:
#                     tokens.extend([left_root, right_root] + temp_suffixes)
#                     compound_found = True
#                     break
#             if not compound_found:
#                 tokens.extend([current_str] + temp_suffixes)
        
#         return tokens

    
#     def encode(self, text):
#         token_ids = []
#         token_ids.append(self.vocab['[BOS]'])

#         text = text.lower()
#         text = self._normalize_text(text)
        
#         for mwe in self.m_word_exp:
#             if mwe in text:
#                 text = text.replace(mwe, mwe.replace(' ','_'))
        
#         words = text.split()
#         for word in words:
#             morphemes = self._segment_word(word)
#             for morpheme in morphemes:
#                 morpheme_id = self.vocab.get(morpheme, self.vocab['[UNK]'])
#                 token_ids.append(morpheme_id)
        
#         token_ids.append(self.vocab['[EOS]'])
#         return token_ids

    
#     def decode(self, token_ids):
#         morpheme_strings = []
#         ctrl_ids = [0,2,3]
        
#         for ids in token_ids:
#             if ids in ctrl_ids:
#                 continue
#             elif ids in self.vocab_inv:
#                 morpheme_strings.append(self.vocab_inv[ids])
        
#         final_string = ""
#         for m in morpheme_strings:
#             m = m.replace('_',' ')

#             if final_string == "":
#                 final_string += m
#             elif m.startswith('-'):
#                 final_string += m.replace('-','')
#             elif final_string.endswith('-'):
#                 final_string = final_string[:-1] + m
#             else:
#                 final_string += " " + m
        
#         return final_string

## RAG PIPELINE

In [5]:
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

class GrammarRAGPipeline:
    def __init__(self, embedding_model_name="all-MiniLM-L6-v2"):
        """
        Initializes the embedding model. 
        Downloads the lightweight Hugging Face model to run locally on your machine.
        """
        print(f"Loading embedding model: {embedding_model_name}...")
        self.embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)
        self.vector_store = None

    def ingest_grammar_book(self, file_path):
        """
        Loads the text, chunks it semantically, and embeds into the FAISS database.
        """
        print(f"Loading grammar text from {file_path}")

        # 1. Load raw text file
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_text=f.read()

        # 2. Cleaning text file
        cleaned_text = re.sub(r'\n\s*\d+\s*\n', '\n', raw_text)
        compact_lines = [line.strip() for line in cleaned_text.split('\n') if line.strip()]
        compact_text = '\n'.join(compact_lines)
        doc = Document(page_content=compact_text, metadata={'source': file_path})

        # 3. Chunking the text file
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size = 1200,
            chunk_overlap = 250,
            separators = [
                r"\n(?=\[)",       # PRIORITY 1: Split right before a bracketed phonetic example (e.g., [cakatkoˊpcĕn])
                r"\n(?=\d+\))",    # PRIORITY 2: Split right before a numbered example (e.g., 158) )
                r"\n(?=[A-Z])",    # PRIORITY 3: Split right before a capitalized English paragraph
                "\n",              # Fallback 1: Any newline
                " ",               # Fallback 2: Spaces
                ""                 # Fallback 3: Character level
            ],
            is_separator_regex=True
        )

        chunks = text_splitter.split_documents([doc])
        print(f"Split Grammar into {len(chunks)} chunks")

        # 4. Embedding
        print("Embedding chunks and building FAISS index...")
        self.vector_store = FAISS.from_documents(chunks, self.embeddings)
        print("Vector database built successfully!")

    def save_database(self, folder_path='atakapa_grammar_faiss'):
        if self.vector_store:
            self.vector_store.save_local(folder_path)
            print(f"Database saved to /{folder_path}")

    def query_grammar(self, question, top_k=3):
        if not self.vector_store:
            print("Error: Vector store not loaded.")
            return

        print(f"\n--- Searching for: '{question}' ---")
        results = self.vector_store.similarity_search(question, k=top_k)
        
        for i, doc in enumerate(results):
            print(f"\n[Result {i+1}]")
            print(doc.page_content)

SyntaxError: invalid syntax (1868945031.py, line 1)

## SLM Trainer

|======================================================================================================|
In order to resolve a UnicodeDecodeError being thrown by the trl library, I had to edit the trl library on my machine, specifically chat_template_utils.py. 

The chat templates follow the basic structure in the example below:

cohere2_chat_template = (_CHAT_TEMPLATES_DIR / "cohere2.jinja").read_text()

"encoding='utf-8'" was added inside of the read_text method of each template in the file
|======================================================================================================|

In [5]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

class AtakapaSLMTrainer:
    def __init__(self, model_id="Qwen/Qwen2.5-0.5B", morph_rules_file='atakapa_morphology_UTF8.xlsx'):
        self.model_id = model_id
        self.morph_rules = pd.read_excel(morph_rules_file)
        
    def _extract_glosses(self):
        """Extracts unique English glosses from the morphology file."""
        # Assuming your excel file has a column named 'gloss' or 'meaning'
        # Update the column name 'gloss' to match your actual file
        raw_glosses = self.morph_rules['gloss_tag'].astype(str).dropna().unique().tolist()
        
        # Format them as special tokens (e.g., "1SG" -> "[1SG]") 
        # Skip if they are already formatted this way in your sheet
        gloss_tokens = [f"[{g.strip()}]" for g in raw_glosses if g.strip()]
        return gloss_tokens

    def setup_model_and_tokenizer(self, custom_morphemes):
        print(f"Loading {self.model_id}...")
        
        # 1. Load Tokenizer & Extract Glosses
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        gloss_tokens = self._extract_glosses()
        
        # Combine morphemes and glosses into the custom vocabulary
        custom_vocab = custom_morphemes + gloss_tokens
        
        # Add to tokenizer
        num_added = self.tokenizer.add_tokens(custom_vocab)
        print(f"Added {num_added} new tokens (Morphemes + Glosses).")
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # 2. Load Base Model
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
            device_map="auto"
        )
        
        # 3. Resize Embeddings to fit the new vocabulary
        self.model.resize_token_embeddings(len(self.tokenizer))
        
        # ---------------------------------------------------------
        # 4. APPLY LoRA (PEFT)
        # ---------------------------------------------------------
        peft_config = LoraConfig(
            r=16,               # Rank of the adapter (16 or 32 is great for vocab expansion)
            lora_alpha=32,      # Scaling factor
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            # Target standard attention layers (Target names vary slightly by model, 
            # these match Llama/Qwen architectures)
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            
            # CRITICAL: Tells PEFT to unfreeze and train the embedding layers!
            modules_to_save=["embed_tokens", "lm_head"] 
        )
        
        # Wrap the model in PEFT
        self.model = get_peft_model(self.model, peft_config)
        self.model.print_trainable_parameters()
        
        return self.model, self.tokenizer

    def train(self, formatted_dataset: Dataset):
        """Initializes the SFTTrainer and starts training."""
        
        # Define Training Hyperparameters
        training_args = SFTConfig(
            output_dir="./atakapa-glosser-model",
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,      # Simulate larger batch sizes on small GPUs
            learning_rate=2e-4,                 # 2e-4 is standard for LoRA, but new embeddings might need slightly higher
            logging_steps=10,
            max_steps=500,                      # Adjust based on dataset size
            save_strategy="epoch",
            num_train_epochs=10,
            optim="paged_adamw_8bit",           # Saves VRAM
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            warmup_ratio=0.05,
            lr_scheduler_type="cosine",
            max_length=256, 
            dataset_text_field="text"
        )
        
        # Initialize TRL's Supervised Fine-Tuning Trainer
        trainer = SFTTrainer(
            model=self.model,
            train_dataset=formatted_dataset,
            peft_config=None, # Already applied via get_peft_model above
            processing_class=self.tokenizer,
            args=training_args,
        )
        
        print("Starting training...")
        trainer.train()
        
        # Save the final adapter and embedding weights
        trainer.model.save_pretrained("./atakapa-glosser-final")
        self.tokenizer.save_pretrained("./atakapa-glosser-final")
        print("Training complete and model saved.")

## Hugging Face Dataset Builder

In [2]:
import pandas as pd
from datasets import Dataset
from AtakapaTokenizer import AtakapaTokenizer

class AtakapaDatasetBuilder:
# ==========================================
# HOW TO USE THIS WITH YOUR EXCEL FILES
# ==========================================
# builder = AtakapaDatasetBuilder()

# 1. Load your files
# texts_df = pd.read_excel('excel_file_name.xlsx')
# examples_df = pd.read_excel('excel_file_name.xlsx')

# 2. Convert to Hugging Face Datasets
# hf_texts = builder.build_from_dataframe(texts_df, atakapa_col='atakapa_column_name', gloss_col='english_column_name')
# hf_examples = builder.build_from_dataframe(examples_df, atakapa_col='atakapa_column_name', gloss_col='english_column_name')

# 3. Combine them into one massive training dataset
# from datasets import concatenate_datasets
# final_training_dataset = concatenate_datasets([hf_texts, hf_examples])

# 4. Pass 'final_training_dataset' directly into the SFTTrainer we wrote earlier!
    def __init__(self):
        # Load your custom tokenizer to ensure perfect train/inference symmetry
        self.tokenizer = AtakapaTokenizer()
        
    def segment_for_training(self, raw_text):
        """Passes training text through your exact tokenizer logic."""
        words = str(raw_text).split()
        segmented_words = []
        for w in words:
            # Using your rollback tokenizer's segmenter
            morphemes = self.tokenizer._segment_word(w)
            segmented_words.append(" ".join(morphemes))
        return " ".join(segmented_words)

    def format_chatml_row(self, segmented_atakapa, english_gloss):
        """Wraps the pair in the exact prompt template the SLM will see in production."""
        # Using the standard ChatML format optimized for Qwen / modern SLMs
        template = (
            f"<|im_start|>user\n"
            f"Gloss this segmented Atakapa: {segmented_atakapa}<|im_end|>\n"
            f"<|im_start|>assistant\n"
            f"{english_gloss}<|im_end|>"
        )
        return template

    def build_from_dataframe(self, df, atakapa_col, gloss_col, is_pre_segmented=False):
        """Converts an Excel dataframe into a Hugging Face Dataset."""
        formatted_texts = []
        
        for _, row in df.iterrows():
            atakapa_text = str(row[atakapa_col]).strip()
            gloss_text = str(row[gloss_col]).strip()
            
            # Skip empty rows
            if not atakapa_text or not gloss_text or atakapa_text == 'nan':
                continue
                
            # If your Excel file already has hyphens/segmentation from the linguist, 
            # we can skip the tokenizer. Otherwise, force it through your tokenizer.
            if not is_pre_segmented:
                atakapa_text = self.segment_for_training(atakapa_text)
            # Wrap in ChatML format    
            chatml_string = self.format_chatml_row(atakapa_text, gloss_text)
            formatted_texts.append(chatml_string)
            
        # Convert to a Hugging Face Dataset object
        return Dataset.from_dict({"text": formatted_texts})

